# EXAMEN FINAL - RECUPERACIÓN DE LA INFORMACIÓN
**Nombre:** Yasid Jimenez Jaramillo

**Fecha:** 15/07/2026

In [3]:
import os
import ast
import pandas as pd
from dotenv import load_dotenv
from IPython.display import display
from google import genai

load_dotenv()
api_key = os.environ.get("GEMINI_API_KEY") 
client = genai.Client(api_key=api_key)

In [2]:
# Testear la conexión
response = client.models.generate_content(
    model='gemini-2.5-flash',
    contents='Holaaa, si funciona la conexión?'
)
print(response.text)

¡Hola! Sí, la conexión funciona perfectamente. Estoy aquí para ayudarte.

¿En qué puedo asistirte hoy?


___
### REQUERIMIENTO A: Preparación del corpus

In [4]:
DATASET_PATH = "dataset/arxiv_data.csv"
N_SAMPLE = 2000
RELEVANT_CATEGORIES = {"cs.LG", "cs.AI", "cs.CV", "cs.CL", "cs.RO", "cs.NE", "cs.IR", "stat.ML"}

def parse_terms(value):
    """Convierte el string de términos en una lista real de Python"""
    try:
        parsed = ast.literal_eval(str(value))
        if isinstance(parsed, list):
            return [str(t).strip() for t in parsed]
    except (ValueError, SyntaxError):
        pass
    return [t.strip() for t in str(value).split(",") if t.strip()]

# 1. Cargar el dataset
print("Cargando el dataset...")
df = pd.read_csv(DATASET_PATH)

# Normalizar nombres de columnas (por si usas el arxiv_data_210930-054931.csv)
df = df.rename(columns={"titles": "title", "summaries": "abstract"})
df["terms_list"] = df["terms"].apply(parse_terms)

# 2. Limpieza de datos
df_clean = df.dropna(subset=["title", "abstract"]).copy()
df_clean["title"] = df_clean["title"].astype(str).str.strip()
df_clean["abstract"] = df_clean["abstract"].astype(str).str.strip()
df_clean = df_clean[(df_clean["title"] != "") & (df_clean["abstract"] != "")]
df_clean = df_clean.drop_duplicates(subset=["title"]).reset_index(drop=True)

# 3. Filtrar por categorías de Computer Science e IA
mask_relevant = df_clean["terms_list"].apply(lambda terms: any(t in RELEVANT_CATEGORIES for t in terms))
df_filtered = df_clean[mask_relevant].reset_index(drop=True)

# 4. Muestreo y preparación final del texto a indexar
df_corpus = df_filtered.sample(n=min(N_SAMPLE, len(df_filtered)), random_state=42).reset_index(drop=True)
df_corpus["doc_id"] = [f"doc_{i}" for i in range(len(df_corpus))]
df_corpus["content"] = df_corpus["title"] + ". " + df_corpus["abstract"]

display(df_corpus[["doc_id", "title", "terms"]].head())

Cargando el dataset...


,doc_id,title,terms
0,doc_0,Sum-Product-Transform Networks: Exploiting Sym...,"['stat.ML', 'cs.LG']"
1,doc_1,A Primal-Dual Subgradient Approachfor Fair Met...,"['cs.LG', 'stat.ML']"
2,doc_2,Adversarial Multi-Source Transfer Learning in ...,"['cs.CV', 'cs.LG', 'eess.IV']"
3,doc_3,An Attractor-Guided Neural Networks for Skelet...,['cs.CV']
4,doc_4,Deep Reinforcement Learning for Autonomous Dri...,"['cs.LG', 'cs.AI', 'cs.RO']"


In [5]:
# Guardamos SOLO los 2000 documentos limpios que vamos a usar en el examen
df_corpus.to_csv("corpus_reducido.csv", index=False)
print("✅ Dataset reducido guardado. Este archivo pesa poquísimo y es el único que subiremos a GitHub.")

✅ Dataset reducido guardado. Este archivo pesa poquísimo y es el único que subiremos a GitHub.


___
### REQUERIMIENTO B: Representación mediante embeddings

In [5]:
!pip install -q sentence-transformers


[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [6]:
import os
import json
import numpy as np
from sentence_transformers import SentenceTransformer

# ======================================================
# Configuración del Modelo Local
# ======================================================
# all-MiniLM-L6-v2 es un estándar de Hugging Face para buscar similitud.
# Genera vectores de 384 dimensiones (más ligero que los 768 de Gemini, pero ultra preciso).
EMBEDDING_MODEL_NAME = "all-MiniLM-L6-v2"
EMBED_DIM = 384

EMB_CACHE_PATH = "embeddings_cache.npy"
IDS_CACHE_PATH = "embeddings_cache_ids.json"

# Inicializar el modelo (la primera vez descargará el archivo de 90MB de internet automáticamente)
print("Cargando el modelo de embeddings local...")
model_local = SentenceTransformer(EMBEDDING_MODEL_NAME)

# ======================================================
# Generación / Carga desde Caché
# ======================================================
if os.path.exists(EMB_CACHE_PATH) and os.path.exists(IDS_CACHE_PATH):
    with open(IDS_CACHE_PATH, "r") as f:
        cached_ids = json.load(f)

    if cached_ids == df_corpus["doc_id"].tolist():
        print("✅ Cargando embeddings desde la caché local...")
        doc_embeddings = np.load(EMB_CACHE_PATH)
    else:
        print("⚠️ El corpus cambió. Regenerando embeddings localmente...")
        # Generar embeddings de forma local con la CPU/GPU de tu máquina
        doc_embeddings = model_local.encode(
            df_corpus["content"].tolist(), 
            show_progress_bar=True,
            batch_size=32
        )
        np.save(EMB_CACHE_PATH, doc_embeddings)
        with open(IDS_CACHE_PATH, "w") as f:
            json.dump(df_corpus["doc_id"].tolist(), f)
else:
    print("🚀 Generando embeddings localmente por primera vez (esto tardará segundos)...")
    doc_embeddings = model_local.encode(
        df_corpus["content"].tolist(), 
        show_progress_bar=True,
        batch_size=32
    )
    np.save(EMB_CACHE_PATH, doc_embeddings)
    with open(IDS_CACHE_PATH, "w") as f:
        json.dump(df_corpus["doc_id"].tolist(), f)

print("\n===================================")
print("Embeddings generados correctamente")
print("===================================")
print(f"Cantidad de documentos : {len(doc_embeddings)}")
print(f"Dimensiones            : {doc_embeddings.shape}")

Cargando el modelo de embeddings local...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

🚀 Generando embeddings localmente por primera vez (esto tardará segundos)...


Batches:   0%|          | 0/63 [00:00<?, ?it/s]


Embeddings generados correctamente
Cantidad de documentos : 2000
Dimensiones            : (2000, 384)


___
### REQUERIMIENTO C: Almacenamiento y búsqueda vectorial

In [9]:
!pip install chromadb

Defaulting to user installation because normal site-packages is not writeable
  Using cached chromadb-1.5.9-cp39-abi3-win_amd64.whl.metadata (5.1 kB)
  Using cached build-1.5.0-py3-none-any.whl.metadata (5.7 kB)
  Using cached uvicorn-0.51.0-py3-none-any.whl.metadata (6.6 kB)
  Using cached opentelemetry_api-1.43.0-py3-none-any.whl.metadata (1.4 kB)
  Using cached opentelemetry_exporter_otlp_proto_grpc-1.43.0-py3-none-any.whl.metadata (2.6 kB)
  Using cached opentelemetry_sdk-1.43.0-py3-none-any.whl.metadata (1.7 kB)
  Using cached pypika-0.51.1-py2.py3-none-any.whl.metadata (51 kB)
  Using cached importlib_resources-7.1.0-py3-none-any.whl.metadata (4.0 kB)
  Using cached kubernetes-36.0.3-py2.py3-none-any.whl.metadata (1.8 kB)
  Using cached pyproject_hooks-1.2.0-py3-none-any.whl.metadata (1.3 kB)
  Using cached requests_oauthlib-2.0.0-py2.py3-none-any.whl.metadata (11 kB)
  Using cached durationpy-0.10-py3-none-any.whl.metadata (340 bytes)
  Using cached flatbuffers-25.12.19-py2.py3-

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [10]:
import chromadb

CHROMA_PATH = "./chroma_db"
COLLECTION_NAME = "arxiv_abstracts"

# 1. Inicializar el cliente persistente (creará la carpeta en tu proyecto)
chroma_client = chromadb.PersistentClient(path=CHROMA_PATH)

# 2. Crear o cargar la colección, especificando distancia coseno
collection = chroma_client.get_or_create_collection(
    name=COLLECTION_NAME,
    metadata={"hnsw:space": "cosine"}
)

# 3. Validar si ya tenemos los datos cargados para no duplicar
existing_ids = set(collection.get(include=[])["ids"]) if collection.count() > 0 else set()
expected_ids = set(df_corpus["doc_id"].tolist())

if expected_ids != existing_ids:
    print("Guardando los vectores y metadatos en ChromaDB...")
    
    # Si había algo viejo que no coincide, lo borramos
    if existing_ids:
        collection.delete(ids=list(existing_ids))
    
    # Poblamos la base de datos con los IDs, vectores, texto y metadatos
    collection.add(
        ids=df_corpus["doc_id"].tolist(),
        embeddings=doc_embeddings.tolist(),
        documents=df_corpus["content"].tolist(),
        metadatas=df_corpus[["title", "terms"]].astype(str).to_dict("records")
    )
    print("✅ Base de datos vectorial poblada exitosamente.")
else:
    print("ℹ️ ChromaDB ya contiene los documentos de este corpus. No es necesario reindexar.")

print(f"Total de documentos en la base de datos vectorial: {collection.count()}")

Guardando los vectores y metadatos en ChromaDB...
✅ Base de datos vectorial poblada exitosamente.
Total de documentos en la base de datos vectorial: 2000


In [11]:
# ======================================================
# Prueba rápida de búsqueda vectorial
# ======================================================
query_prueba = "What are the main applications of Graph Neural Networks?"

# Convertimos la consulta a un vector de 384 dimensiones usando nuestro modelo local
query_emb = model_local.encode([query_prueba]).tolist()

# Buscamos los 3 abstractos más similares
res = collection.query(query_embeddings=query_emb, n_results=3)

print(f"Resultados de búsqueda para: '{query_prueba}'\n")
for idx, (doc_id, dist, meta) in enumerate(zip(res["ids"][0], res["distances"][0], res["metadatas"][0])):
    similitud = 1 - dist  # Convertimos la distancia coseno en similitud
    print(f" {idx+1}. [{doc_id}] Similitud: {similitud:.4f}")
    print(f"    Título: {meta['title']}\n")

Resultados de búsqueda para: 'What are the main applications of Graph Neural Networks?'

 1. [doc_1554] Similitud: 0.6123
    Título: Hierarchical Bipartite Graph Convolution Networks

 2. [doc_384] Similitud: 0.5605
    Título: Exploiting Edge Features in Graph Neural Networks

 3. [doc_947] Similitud: 0.5532
    Título: GraphNorm: A Principled Approach to Accelerating Graph Neural Network Training



___
### REQUERIMIENTO D: Recuperación

In [12]:
from sentence_transformers import CrossEncoder

# ======================================================
# Configuración del Cross-Encoder para Re-ranking
# ======================================================
print("Cargando modelo Cross-Encoder para re-ranking...")
# Este es el estándar de la industria para re-ranking ligero y rápido
reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def recuperar_documentos(query: str, top_k: int = 5, fetch_k: int = 20) -> list[dict]:
    """
    Pipeline de recuperación en dos fases:
    1. Recupera 'fetch_k' candidatos usando embeddings y ChromaDB.
    2. Re-ordena los candidatos usando el Cross-Encoder y devuelve los 'top_k' mejores.
    """
    
    # --- FASE 1: Búsqueda Vectorial (Bi-Encoder) ---
    query_emb = model_local.encode([query]).tolist()
    results = collection.query(query_embeddings=query_emb, n_results=fetch_k)

    candidatos = []
    for doc_id, distance, document, metadata in zip(
        results["ids"][0], results["distances"][0], results["documents"][0], results["metadatas"][0]
    ):
        candidatos.append({
            "doc_id": doc_id,
            "similitud_vectorial": round(1 - distance, 4),
            "titulo": metadata.get("title", ""),
            "categorias": metadata.get("terms", ""),
            "fragmento": document,
        })

    # --- FASE 2: Re-ranking (Cross-Encoder) ---
    # Armamos pares de [Pregunta, Fragmento_del_Documento]
    pares_rerank = [[query, doc["fragmento"]] for doc in candidatos]
    
    # El modelo predice un score para cada par
    scores = reranker.predict(pares_rerank)
    
    # Guardamos el score en nuestra lista de diccionarios
    for i, doc in enumerate(candidatos):
        doc["score_rerank"] = float(scores[i])
        
    # Ordenamos la lista basándonos en el nuevo score de mayor a menor
    candidatos_ordenados = sorted(candidatos, key=lambda x: x["score_rerank"], reverse=True)
    
    # Devolvemos solo los 'top_k' solicitados
    return candidatos_ordenados[:top_k]

print("✅ Función de recuperación con re-ranking lista.")

Cargando modelo Cross-Encoder para re-ranking...


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

✅ Función de recuperación con re-ranking lista.


In [13]:
# ======================================================
# Prueba del motor de Recuperación + Re-ranking
# ======================================================
consulta_ejemplo = "How is reinforcement learning used in robotics?"

print(f"Buscando evidencia para: '{consulta_ejemplo}'...\n")
evidencias = recuperar_documentos(consulta_ejemplo, top_k=3)

for idx, ev in enumerate(evidencias):
    print(f"--- Top {idx+1} [{ev['doc_id']}] ---")
    print(f"Score Re-rank: {ev['score_rerank']:.2f} | Similitud Vec: {ev['similitud_vectorial']:.4f}")
    print(f"Título: {ev['titulo']}\n")

Buscando evidencia para: 'How is reinforcement learning used in robotics?'...

--- Top 1 [doc_157] ---
Score Re-rank: 6.90 | Similitud Vec: 0.5944
Título: Low Dimensional State Representation Learning with Reward-shaped Priors

--- Top 2 [doc_1876] ---
Score Re-rank: 6.31 | Similitud Vec: 0.5534
Título: Creativity in Robot Manipulation with Deep Reinforcement Learning

--- Top 3 [doc_1446] ---
Score Re-rank: 5.92 | Similitud Vec: 0.5311
Título: A Brief Survey of Deep Reinforcement Learning



___
### REQUERIMIENTO E: Generación aumentada por recuperación

In [18]:
import time
from google.genai import types
from tenacity import retry, wait_exponential, stop_after_attempt, retry_if_exception_type
from google.genai.errors import ServerError

# Usamos el modelo rápido y económico recomendado para tareas de texto
GENERATION_MODEL = "gemini-2.5-flash"

# ======================================================
# Instrucciones del Sistema (System Instructions)
# ======================================================
SYSTEM_INSTRUCTION = (
    "Eres un asistente experto en literatura científica que responde preguntas usando "
    "exclusivamente los fragmentos de contexto (abstracts de arXiv) que se te entregan. "
    "Reglas:\n"
    "1. Responde ÚNICAMENTE con la información contenida en el contexto proporcionado.\n"
    "2. Si el contexto no contiene información suficiente para responder con confianza, dilo "
    "explícitamente en tu respuesta (por ejemplo: 'El corpus recuperado no contiene información "
    "suficiente para responder esta pregunta') en vez de inventar datos.\n"
    "3. Cuando uses información de un fragmento, cita su identificador entre corchetes, "
    "por ejemplo [doc_12].\n"
    "4. Si la evidencia proviene de varios documentos, intenta integrarlos en una respuesta coherente.\n"
    "5. Responde en el mismo idioma en el que fue formulada la pregunta del usuario."
)

def build_prompt(query: str, evidencias: list[dict]) -> str:
    """Inyecta los fragmentos recuperados en el prompt."""
    contexto = "\n\n".join(
        f"[{e['doc_id']}] Título: {e['titulo']}\n"
        f"Categorías: {e['categorias']}\n"
        f"Abstract: {e['fragmento']}"
        for e in evidencias
    )
    return (
        f"Contexto recuperado del corpus (arXiv abstracts):\n\n{contexto}\n\n"
        f"---\n\nPregunta del usuario: {query}"
    )

# Añadimos un decorador que reintentará hasta 4 veces si hay un error del servidor (503),
# esperando 2s, 4s y 8s entre intentos.
@retry(
    retry=retry_if_exception_type(ServerError),
    wait=wait_exponential(multiplier=2, min=2, max=10),
    stop=stop_after_attempt(4),
    before_sleep=lambda retry_state: print(f"⚠️ Servidor ocupado. Reintentando en {retry_state.next_action.sleep}s...")
)
def generar_respuesta(query: str, evidencias: list[dict]) -> str:
    """Llama a Gemini pasándole el contexto y la pregunta."""
    prompt = build_prompt(query, evidencias)
    
    response = client.models.generate_content(
        model=GENERATION_MODEL,
        contents=prompt,
        config=types.GenerateContentConfig(
            system_instruction=SYSTEM_INSTRUCTION,
            temperature=0.2, # Muy bajo para evitar alucinaciones
        ),
    )
    return response.text

def pipeline_rag(query: str, k: int = 5) -> tuple[str, list[dict]]:
    """Ejecuta el flujo RAG completo: Recuperación en 2 fases + Generación."""
    evidencias = recuperar_documentos(query, top_k=k)
    respuesta = generar_respuesta(query, evidencias)
    return respuesta, evidencias

print("✅ Funciones de generación RAG listas (con reintentos automáticos).")

✅ Funciones de generación RAG listas (con reintentos automáticos).


In [17]:
!pip install tenacity

Defaulting to user installation because normal site-packages is not writeable



[notice] A new release of pip is available: 26.1.1 -> 26.1.2
[notice] To update, run: python.exe -m pip install --upgrade pip


In [19]:
# ======================================================
# Prueba del Pipeline RAG Completo
# ======================================================

print("--- PRUEBA 1: Pregunta dentro del dominio ---")
pregunta_valida = "Recent advances in diffusion models for image generation."
respuesta_1, ev_1 = pipeline_rag(pregunta_valida)
print(f"Pregunta: {pregunta_valida}\n")
print(f"Respuesta Gemini:\n{respuesta_1}\n")


print("--- PRUEBA 2: Pregunta fuera del dominio (Comprobando Regla #2) ---")
pregunta_trampa = "¿Cuál es la capital de Francia y cómo hacer un pastel de chocolate?"
respuesta_2, ev_2 = pipeline_rag(pregunta_trampa)
print(f"Pregunta: {pregunta_trampa}\n")
print(f"Respuesta Gemini:\n{respuesta_2}")

--- PRUEBA 1: Pregunta dentro del dominio ---
Pregunta: Recent advances in diffusion models for image generation.

Respuesta Gemini:
El corpus recuperado no contiene información suficiente para responder esta pregunta sobre los avances recientes en modelos de difusión para la generación de imágenes.

Sin embargo, uno de los documentos menciona un esquema de difusión para el procesamiento de imágenes, específicamente para el "denoising" (eliminación de ruido) de imágenes digitales a color mediante un esquema de difusión geométrica mejorado [doc_1967]. Este esquema utiliza una difusión vectorial para controlar el suavizado y considera la geometría de las imágenes a color, introduciendo la fuerza del borde de color para detener la difusión a través de los bordes cromáticos [doc_1967]. Este método se enfoca en la preservación de bordes durante el denoising [doc_1967].

--- PRUEBA 2: Pregunta fuera del dominio (Comprobando Regla #2) ---
Pregunta: ¿Cuál es la capital de Francia y cómo hacer 

___
### REQUERIMIENTO F: Presentación de evidencias


In [20]:
import pandas as pd
from IPython.display import display, Markdown

# ======================================================
# Función para presentar resultados de forma visual
# ======================================================
def presentar_resultados(query: str, respuesta: str, evidencias: list[dict]):
    """
    Muestra la consulta, la respuesta y genera un DataFrame con las evidencias
    para facilitar la auditoría de los resultados.
    """
    # 1. Mostrar Consulta y Respuesta con Markdown
    display(Markdown(f"### 🔎 Consulta\n> **{query}**"))
    display(Markdown(f"### 🤖 Respuesta Generada\n{respuesta}"))
    
    # 2. Crear y mostrar el DataFrame con las métricas de las evidencias
    display(Markdown("### 📚 Evidencias Utilizadas (Tabla Resumen)"))
    
    # Convertimos la lista de diccionarios en un DataFrame
    df_evidencias = pd.DataFrame(evidencias)
    
    # Seleccionamos las columnas clave para la tabla visual
    columnas_mostrar = ["doc_id", "score_rerank", "similitud_vectorial", "titulo", "categorias"]
    df_tabla = df_evidencias[columnas_mostrar].copy()
    
    # Renderizamos la tabla (se verá con formato de tabla nativa en VS Code)
    display(df_tabla)
    
    # 3. Mostrar los textos completos debajo (opcional pero útil)
    # Los DataFrames suelen truncar los textos largos, así que imprimimos el abstract completo
    display(Markdown("#### 📝 Fragmentos Completos Recuperados"))
    for ev in evidencias:
        print(f"[{ev['doc_id']}] TÍTULO: {ev['titulo']}")
        print(f"CATEGORÍAS: {ev['categorias']}")
        # Imprimimos el abstract (puedes truncarlo si es muy largo, pero para el examen es mejor completo)
        print(f"ABSTRACT: {ev['fragmento']}")
        print("-" * 80 + "\n")

print("✅ Función de presentación visual lista.")

✅ Función de presentación visual lista.


In [21]:
# ======================================================
# Prueba visual final (End-to-End en el Notebook)
# ======================================================
consulta_ejemplo = "What are the main applications of Graph Neural Networks?"

# Ejecutamos nuestro pipeline
respuesta, evidencias = pipeline_rag(consulta_ejemplo, k=5)

# Mostramos los resultados usando nuestra nueva función
presentar_resultados(consulta_ejemplo, respuesta, evidencias)

### 🔎 Consulta
> **What are the main applications of Graph Neural Networks?**

### 🤖 Respuesta Generada
Las principales aplicaciones de las Redes Neuronales Gráficas (GNNs) incluyen:

*   Representaciones relacionales [doc_1554].
*   Modelado de dominios de datos irregulares como nubes de puntos y grafos sociales [doc_1554].
*   Quimioinformática [doc_998].
*   Sistemas de respuesta a preguntas [doc_998].
*   Sistemas de recomendación [doc_998].
*   Conjuntos de datos de redes de citas (Cora, Citeseer y Pubmed) [doc_1482].
*   Conjuntos de datos de interacción proteína-proteína [doc_1482].

### 📚 Evidencias Utilizadas (Tabla Resumen)

,doc_id,score_rerank,similitud_vectorial,titulo,categorias
0,doc_1554,7.135324,0.6123,Hierarchical Bipartite Graph Convolution Networks,"['cs.LG', 'cs.CV', 'stat.ML']"
1,doc_998,5.100887,0.5081,Constant Time Graph Neural Networks,"['cs.LG', 'stat.ML']"
2,doc_1482,3.199501,0.5224,Graph Attention Networks,"['stat.ML', 'cs.AI', 'cs.LG', 'cs.SI']"
3,doc_189,3.016830,0.5052,Beyond Low-frequency Information in Graph Conv...,"['cs.LG', 'cs.SI']"
4,doc_881,2.775743,0.5229,Design Space for Graph Neural Networks,"['cs.LG', 'cs.AI', 'cs.SI']"


#### 📝 Fragmentos Completos Recuperados

[doc_1554] TÍTULO: Hierarchical Bipartite Graph Convolution Networks
CATEGORÍAS: ['cs.LG', 'cs.CV', 'stat.ML']
ABSTRACT: Hierarchical Bipartite Graph Convolution Networks. Recently, graph neural networks have been adopted in a wide variety of
applications ranging from relational representations to modeling irregular data
domains such as point clouds and social graphs. However, the space of graph
neural network architectures remains highly fragmented impeding the development
of optimized implementations similar to what is available for convolutional
neural networks. In this work, we present BiGraphNet, a graph neural network
architecture that generalizes many popular graph neural network models and
enables new efficient operations similar to those supported by ConvNets. By
explicitly separating the input and output nodes, BiGraphNet: (i) generalizes
the graph convolution to support new efficient operations such as coarsened
graph convolutions (similar to strided convolution in convnets)

___
### REQUERIMIENTO G: Interfaz web conversacional

![Interfaz web conversacional - Query Funcional](images/1.png)

![Interfaz web conversacional - Query Invalida](images/2.png)

___
### REQUERIMIENTO H: Despliegue en la nube

___
### REQUERIMIENTO I: Evaluación del sistema y de la generación